In [ ]:
# Preparation cell: If not already installed, run "pip install fr24sdk"

import os
from fr24sdk.client import Client

# The FR24 account page will provide an access token that is only visible once for security purposes, though it can be used indefinitely.
os.environ["FR24_API_TOKEN"] = "long_token_code_here"

# You can run the following to make sure the token worked correctly, just for your own sanity. 
with Client(api_token=os.environ["FR24_API_TOKEN"]) as client:
    print(client.usage.get())

print("Ready to pull data.")


In [ ]:
# This method appends to a per-airline MASTER CSV that can be resumed at any time. However, use this for one airline at a time.
# As explained in the paper, this code also assumes a preexisting masterlist of aircraft registration IDs (also available through FR24's databases).

import os, json, time, random, pathlib
from datetime import datetime, date, timedelta, timezone
import pandas as pd
import requests

# Configuration settings (per airline / run / etc.)
EXCEL_PATH  = r"C:\Users\file_path_of_your_choice\ExampleDataPull1.xlsx"
SHEET_NAME  = "Qatar"   # this can be a string (sheet name), an int (e.g., 0 for first sheet), or None (auto first sheet) if you have multiple.
REG_COL     = "Registration"  # change if your column header differs (e.g., "Registration", "CraftID", "Reg", etc.)
OUT_PREFIX  = "qatar"   # This names the output files, change to your situation: we used "fedex", "ups", "qatar", and "emirates"

START_DATE       = "2025-01-01"   # inclusive from 00:00:00 of the date you choose. FR24 plans have different date ranges allowed. 
END_DATE         = "2025-12-31"   # inclusive to 11:59:59 of the data you choose. If these are outside of the allowed date ranges, the pull will fail.

# These can edited, but I found this config circumvented FR24's data rate limit errors and still pulled entries quickly enough. 
# If too many errors occur, the whole run can be terminated, so be cognizant of rate limits before leaving the pull unattended.
BATCH_SIZE   = 15     # regs per request
LIMIT        = 20     # FIRST PAGE ONLY
BASE_DELAY   = 2.0
MAX_RETRIES  = 6

# Budget caps if you want to only pull a certain amount of credits-worth of data, for some specific reason. 
PER_DAY_CREDIT_CAP = None          # stops per-day pulls if vakue is reached, will continue to next day (set None to disable)
TOTAL_CREDIT_CAP   = None          # stops the whole pull if value is reached (set None to disable)

# ========= AUTH / OUTPUT =========
# OUT_DIR sets the output location for the per-day AND master CSV files. It SHOULD create a folder if it doesn't exist already, but safest to just use
# a preexisting folder. The MASTER_CSV line can be renamed to whatever else, but this is the compiled file for all daily pulls, for reference. This does
# NOT pull entries twice (one for daily, one for master), it simply copies the daily over to a master, so this does not increase credit cost.
# Leave the OUT_DIR.mkdir line alone.

OUT_DIR = pathlib.Path(r"C:\Users\folder_name") 
MASTER_CSV = OUT_DIR / f"{OUT_PREFIX}_flight_summary_MASTER.csv" 
OUT_DIR.mkdir(parents=True, exist_ok=True) 


TOKEN = (globals().get("api_token") or
         os.getenv("FR24_API_TOKEN") or
         os.getenv("FLIGHTRADAR_API_KEY"))
assert TOKEN, "Set `api_token = '...'` (or FR24_API_TOKEN / FLIGHTRADAR_API_KEY) before running."

BASE = "https://fr24api.flightradar24.com/api"
ses = requests.Session()
ses.headers.update({
    "Authorization": f"Bearer {TOKEN}",
    "Accept-Version": "v1",
    "Accept": "application/json",
    "User-Agent": f"fr24-{OUT_PREFIX}-range-runner/1.1"
})



# The lines below are mostly just functional lines and should generally be left unaltered unless you really need something to be changed. 
# They will let you know the overall progress, an estimate of credits used for the pull, optimize data management as entries are pulled, etc. 
# Additionally, state.json files are created for each day so that entries will not be duplicated if you make a mistake and need to pull something again. 
# There are print() functions at the end that will tell you these updates in the Python environment. Small notes are included throughout. 


def inferred_rate_per_row(day_str: str) -> int:
    try:
        d = datetime.fromisoformat(day_str).replace(tzinfo=timezone.utc)
    except Exception:
        d = datetime.strptime(day_str, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    age = (datetime.now(timezone.utc) - d).days
    return 3 if age <= 30 else 6  # Light pricing heuristic

def _resolve_sheet_name(xls: pd.ExcelFile, sheet):
    sheets = xls.sheet_names
    if sheet is None:
        return sheets[0]
    if isinstance(sheet, int):
        if sheet < 0 or sheet >= len(sheets):
            raise ValueError(f"SHEET_NAME index {sheet} is out of range. Available sheets: {sheets}")
        return sheets[sheet]
    # string
    if sheet not in sheets:
        raise ValueError(f"Sheet '{sheet}' not found. Available sheets: {sheets}")
    return sheet

def read_regs(xlsx_path: str, sheet, col: str) -> list[str]:
    """Load registrations from an Excel sheet. `sheet` may be a name (str), index (int), or None (first sheet)."""
    xls = pd.ExcelFile(xlsx_path)
    target_sheet = _resolve_sheet_name(xls, sheet)
    df = pd.read_excel(xls, sheet_name=target_sheet, dtype=str)
    print(f"Using sheet: '{target_sheet}'. Columns: {list(df.columns)}")
    if col not in df.columns:
        raise ValueError(f"Column '{col}' not found in sheet '{target_sheet}'. "
                         f"Found columns: {list(df.columns)}")
    series = (df[col].dropna().astype(str).str.strip().str.upper().str.replace(" ", "", regex=False))
    mask   = series.str.fullmatch(r"[A-Z0-9\-\/]{3,}")
    regs = sorted(series[mask].unique().tolist())
    if not regs:
        raise ValueError(f"No valid registrations parsed from column '{col}' in sheet '{target_sheet}'.")
    return regs

def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def backoff_get(url, params):
    attempt = 0
    while True:
        attempt += 1
        r = ses.get(url, params=params, timeout=60)
        if r.status_code < 400:
            time.sleep(BASE_DELAY + random.uniform(0, 0.4))
            try:
                return r.json(), True
            except Exception:
                return {}, True
        if r.status_code in (429, 500, 502, 503, 504):
            ra = r.headers.get("Retry-After")
            if ra:
                try: wait = float(ra)
                except Exception: wait = (2 ** min(attempt, 6)) + random.uniform(0, 1.0)
            else:
                wait = (2 ** min(attempt, 6)) + random.uniform(0, 1.0)
            msg = r.text[:200].replace("\n"," ")
            print(f"    BACKOFF {r.status_code}: wait {wait:.1f}s (attempt {attempt}/{MAX_RETRIES}). {msg}")
            time.sleep(wait)
            if attempt < MAX_RETRIES:
                continue
            print("    GAVE UP after max retries.")
            return None, False
        print(f"    HTTP {r.status_code}: {r.url}\n    {r.text[:200].replace(chr(10),' ')}")
        return None, False

def save_per_day_csv(csv_path, rows):
    if not rows:
        return 0
    out = pd.json_normalize(rows, sep=".")
    keep = [c for c in ["flight","registration","orig_icao","dest_icao","datetime_takeoff","datetime_landed","fr24_id"] if c in out.columns]
    out  = out[keep] if keep else out
    if csv_path.exists():
        prev = pd.read_csv(csv_path)
        merged = pd.concat([prev, out], ignore_index=True)
        if "fr24_id" in merged.columns:
            merged = merged.drop_duplicates(subset=["fr24_id"])
        merged.to_csv(csv_path, index=False, encoding="utf-8-sig")
        return len(merged)
    else:
        if "fr24_id" in out.columns:
            out = out.drop_duplicates(subset=["fr24_id"])
        out.to_csv(csv_path, index=False, encoding="utf-8-sig")
        return len(out)

def update_master(master_csv, new_rows):
    if not new_rows:
        return 0
    nr = pd.json_normalize(new_rows, sep=".")
    keep = [c for c in ["flight","registration","orig_icao","dest_icao","datetime_takeoff","datetime_landed","fr24_id"] if c in nr.columns]
    nr  = nr[keep] if keep else nr
    if master_csv.exists():
        old = pd.read_csv(master_csv)
        merged = pd.concat([old, nr], ignore_index=True)
        if "fr24_id" in merged.columns:
            merged = merged.drop_duplicates(subset=["fr24_id"])
        merged.to_csv(master_csv, index=False, encoding="utf-8-sig")
        return len(merged)
    else:
        if "fr24_id" in nr.columns:
            nr = nr.drop_duplicates(subset=["fr24_id"])
        nr.to_csv(master_csv, index=False, encoding="utf-8-sig")
        return len(nr)

def _read_csv_str(path, usecols=None):
    return pd.read_csv(path, dtype=str, low_memory=False, usecols=usecols)


# Load all registrations once
regs_all = read_regs(EXCEL_PATH, SHEET_NAME, REG_COL)
batches_all = [list(b) for b in chunks(regs_all, BATCH_SIZE)]
print(f"Regs: {len(regs_all)}  |  Batches (@{BATCH_SIZE}): {len(batches_all)}")

# Iterate dates
d0 = datetime.strptime(START_DATE, "%Y-%m-%d").date()
d1 = datetime.strptime(END_DATE,   "%Y-%m-%d").date()
assert d1 >= d0, "END_DATE must be >= START_DATE"

total_rows_returned = 0
total_credit_est    = 0

for day in (d0 + timedelta(n) for n in range((d1 - d0).days + 1)):
    day_str = day.isoformat()
    if day_str < PLAN_START_DATE:
        print(f"\n{day_str}: skip — earlier than PLAN_START_DATE={PLAN_START_DATE}")
        continue

    rate = inferred_rate_per_row(day_str)   # 3 if <=30 days old, else 6
    row_cap = (PER_DAY_CREDIT_CAP // rate) if PER_DAY_CREDIT_CAP else float("inf")

    CSV_PATH   = OUT_DIR / f"{OUT_PREFIX}_flight_summary_{day_str}.csv"
    STATE_PATH = OUT_DIR / f"{OUT_PREFIX}_flight_summary_{day_str}.state.json"

    # load/merge state
    state = {"date": day_str, "done_batches": [], "offsets": {}}
    if STATE_PATH.exists():
        try:
            s = json.loads(STATE_PATH.read_text(encoding="utf-8"))
            if s.get("date") == day_str:
                state.update({k: s.get(k, state.get(k, {})) for k in ["done_batches","offsets"]})
        except Exception:
            pass
    done_batches = set(state.get("done_batches", []))
    offsets = dict(state.get("offsets", {}))
    sigs    = [",".join(b) for b in batches_all]

    # skip batches where page 1 was already fetched (offset >= LIMIT)
    skip_first_page = {sig for sig in sigs if (sig in done_batches or int(offsets.get(sig, 0)) >= LIMIT)}
    todo = [(batch, sig) for batch, sig in zip(batches_all, sigs) if sig not in skip_first_page]

    # local dedupe for writing
    known_ids = set()
    if CSV_PATH.exists():
        try:
            prev = _read_csv_str(CSV_PATH, usecols=["fr24_id"])
            if "fr24_id" in prev.columns:
                known_ids = set(prev["fr24_id"].dropna().astype(str))
        except Exception:
            pass

    rows_returned_today = 0
    new_rows_today = []

    print(f"\n{day_str}: remaining batches {len(todo)} (skip {len(skip_first_page)}/{len(sigs)}); rate≈{rate} credits/row; cap≈{row_cap if row_cap!=float('inf') else '∞'} rows")

    if not todo:
        # keep master up to date even if day is already complete
        if CSV_PATH.exists():
            already = _read_csv_str(CSV_PATH)
            _ = update_master(MASTER_CSV, already.to_dict(orient="records"))
            print("  COMPLETE already. Master updated.")
        else:
            print("  COMPLETE already (no per-day CSV found).")
        continue

    for idx, (reg_batch, sig) in enumerate(todo, 1):
        # Total-run budget guard
        if TOTAL_CREDIT_CAP is not None and (total_credit_est + rows_returned_today * rate) >= TOTAL_CREDIT_CAP:
            print(f"  🔒 TOTAL credit cap reached; stopping range.")
            break
        # Per-day budget guard
        if rows_returned_today >= row_cap:
            print(f"  🔒 DAY credit cap reached (~{rows_returned_today * rate} credits); pausing {day_str}.")
            break

        # first page only
        url = f"{BASE}/flight-summary/light"
        params = {
            "flight_datetime_from": f"{day_str}T00:00:00",
            "flight_datetime_to":   f"{day_str}T23:59:59",
            "registrations": ",".join(reg_batch),
            "limit": LIMIT,
            "offset": 0
        }
        j, ok = backoff_get(url, params)
        page = []
        if ok and j is not None:
            page = j.get("data", j) if isinstance(j, dict) else j
            if not isinstance(page, list):
                page = []
            # normalize reg field
            for row in page:
                if not row.get("registration") and row.get("reg"):
                    row["registration"] = row["reg"]

        got = len(page)
        rows_returned_today += got

        # mark page-1 done for this batch (offset >= LIMIT)
        offsets[sig] = max(int(offsets.get(sig, 0)), LIMIT)
        done_batches.add(sig)
        state["offsets"] = offsets
        state["done_batches"] = sorted(done_batches)
        STATE_PATH.write_text(json.dumps(state, indent=2), encoding="utf-8")

        kept = 0
        for row in page:
            fid = str(row.get("fr24_id") or "")
            if fid and fid not in known_ids:
                known_ids.add(fid)
                new_rows_today.append(row)
                kept += 1

        print(f"  [{idx}/{len(todo)}] {len(reg_batch):2d} regs → {got:2d} rows (saved {kept:2d}) — page 1 done")

        # per-day cap check
        if rows_returned_today >= row_cap:
            print(f"  🔒 DAY credit cap reached (~{rows_returned_today * rate} credits); pausing {day_str}.")
            break

    # Save per-day CSV + update master
    if new_rows_today:
        out = pd.json_normalize(new_rows_today, sep=".")
        keep = [c for c in ["flight","registration","orig_icao","dest_icao","datetime_takeoff","datetime_landed","fr24_id"] if c in out.columns]
        out  = out[keep] if keep else out
        if CSV_PATH.exists():
            prev = _read_csv_str(CSV_PATH)
            merged = pd.concat([prev, out], ignore_index=True)
            if "fr24_id" in merged.columns:
                merged = merged.drop_duplicates(subset=["fr24_id"])
            merged.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
            rows_after = len(merged)
        else:
            if "fr24_id" in out.columns:
                out = out.drop_duplicates(subset=["fr24_id"])
            out.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
            rows_after = len(out)

        _ = update_master(MASTER_CSV, new_rows_today)
        print(f"  Saved day → {CSV_PATH} (unique now: {rows_after}); MASTER updated.")
    else:
        print("  No new rows to save for this run (day complete or empty page).")

    day_credits = rows_returned_today * rate
    total_rows_returned += rows_returned_today
    total_credit_est    += day_credits
    print(f"  ≈{day_credits} credits for {day_str} this run (rate {rate}/row).")

print(f"\n=== RANGE SUMMARY ({OUT_PREFIX}) ===")
print(f"Rows returned across this run: {total_rows_returned}")
print(f"Approx credits across this run: ~{total_credit_est}")
print(f"Master CSV: {MASTER_CSV}  ({'exists' if MASTER_CSV.exists() else 'not created'})")
print("Re-running is safe: page-1 work per day is remembered in each .state.json and will be skipped.")
